# Deploy a Model as an API (FastAPI + Docker)

A trained model is useless sitting on disk. To *serve* it we wrap it in a small web API so any client — a website, a mobile app, another microservice — can POST a feature vector and get a prediction back over HTTP.

This notebook walks the full path, end to end and **fully offline**:

1. **Train** a small scikit-learn classifier inside a `Pipeline` (scaler + model) and persist the *whole pipeline* with `joblib`.
2. **Generate `app.py`** — a FastAPI service that loads the pipeline at startup, validates input with a Pydantic schema, and exposes `POST /predict` and `GET /health`.
3. **Prove it works** by exercising the app *in-process* with `TestClient` — no blocking server, so the notebook actually finishes.
4. **Containerize** it — write a `Dockerfile` and `requirements.txt` so the exact same service runs anywhere.

> **Why `TestClient` instead of a live server?** `uvicorn.run(...)` blocks forever waiting for requests — a notebook cell running it would hang and never complete. `TestClient` (Starlette's, re-exported by FastAPI) drives the *same* ASGI app object in the same process, so we get real request/response behavior with zero networking and zero hanging.

In [ ]:
import os                                              # filesystem: build the api_service folder
import json                                            # pretty-print the JSON the API returns
import numpy as np                                     # numeric arrays for the sample feature row
import joblib                                          # persist/reload the fitted sklearn pipeline
from sklearn.datasets import load_breast_cancer        # small, real, offline classification dataset
from sklearn.model_selection import train_test_split   # honest train/test split
from sklearn.pipeline import Pipeline                  # chains preprocessing + model into ONE object
from sklearn.preprocessing import StandardScaler       # zero-mean/unit-variance feature scaling
from sklearn.linear_model import LogisticRegression    # the classifier we will serve

# Everything the notebook generates (model artifact, app.py, Dockerfile, requirements.txt)
# lands in ONE subfolder next to this notebook, so the whole "service" is self-contained.
SERVICE_DIR = os.path.join(os.getcwd(), "api_service")
os.makedirs(SERVICE_DIR, exist_ok=True)                # exist_ok=True -> safe to re-run the cell
MODEL_PATH = os.path.join(SERVICE_DIR, "model.joblib") # where the fitted pipeline is saved

print("service dir:", SERVICE_DIR)

## 1. Train a model *inside a Pipeline*

The single most important deployment idea: **preprocessing must travel with the model.**

At training time we scale the features (`StandardScaler`) before fitting `LogisticRegression`. At serving time the incoming raw feature vector must be scaled *with the exact same statistics* the scaler learned during training. If you save only the bare model and forget the scaler — or re-fit a scaler on serving data — predictions silently go wrong.

A `Pipeline` bundles the scaler and the model into one object. Calling `.predict(...)` on it runs *scale-then-classify* atomically. We persist that whole pipeline, so the deployed service can never drift out of sync with its preprocessing.

In [ ]:
# load_breast_cancer: 569 samples, 30 numeric features, binary target (malignant=0 / benign=1).
# Ships with sklearn, so this is 100% offline.
data = load_breast_cancer()
X, y = data.data, data.target                          # X: (569, 30) floats, y: (569,) 0/1 labels
FEATURE_NAMES = list(data.feature_names)               # keep names -> the API schema documents them
N_FEATURES = X.shape[1]                                 # 30; the API will require exactly this many

# Hold out 25% for an honest accuracy estimate. stratify=y keeps the class ratio in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# The Pipeline = preprocessing + estimator as a SINGLE fitted object.
#   step 'scaler' : learns per-feature mean/std on TRAIN, applies (x - mean)/std
#   step 'clf'    : logistic regression fitted on the scaled features
# max_iter bumped so the solver comfortably converges on 30 scaled features.
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=10000, random_state=42)),
])
pipeline.fit(X_train, y_train)                          # fits BOTH steps in the right order

# Sanity check: how good is the thing we are about to ship?
train_acc = pipeline.score(X_train, y_train)
test_acc = pipeline.score(X_test, y_test)
print(f"features: {N_FEATURES}   train acc: {train_acc:.3f}   test acc: {test_acc:.3f}")

## 2. Persist the whole pipeline with `joblib`

`joblib.dump` serializes the fitted pipeline — learned scaler statistics *and* model coefficients — to a single `.joblib` file. `joblib` is preferred over plain `pickle` for scikit-learn objects because it stores large NumPy arrays efficiently.

The service will `joblib.load` this exact artifact at startup. Because it's the *whole pipeline*, the served predictions are guaranteed to use the same preprocessing as training.

In [ ]:
# Serialize the fitted pipeline to disk. This one file IS the deployable model artifact.
joblib.dump(pipeline, MODEL_PATH)
print("saved model artifact ->", MODEL_PATH)
print("size on disk:", os.path.getsize(MODEL_PATH), "bytes")

# Round-trip check: reload it and confirm predictions match the in-memory pipeline exactly.
reloaded = joblib.load(MODEL_PATH)
assert np.array_equal(reloaded.predict(X_test), pipeline.predict(X_test)), "reloaded model differs!"
print("reload round-trip OK: predictions identical")

## 3. Generate `app.py` (the FastAPI service)

We write the service source as a string to `api_service/app.py`. In production this file is what you'd hand to `uvicorn`. Its responsibilities:

- **Load once at startup.** `joblib.load` the pipeline a single time when the module imports — not per request — so every prediction is fast.
- **Validate input.** A Pydantic `PredictRequest` model declares the request body as a list of floats and rejects the wrong length *before* it ever reaches the model. Bad input gets a clean `422`, not a 500 crash.
- **`POST /predict`** returns the predicted class, the class probabilities, and a `model_version` tag.
- **`GET /health`** is a cheap liveness probe (load balancers / Kubernetes hit it to know the service is up).

The code inside the string is fully commented, exactly like the rest of the notebook.

In [ ]:
# The service source. We embed N_FEATURES / MODEL_VERSION via an f-string so app.py is
# self-consistent with the model we just trained. All literal { } for dicts are doubled {{ }}
# so the f-string does not treat them as substitution fields.
MODEL_VERSION = "1.0.0"
app_source = f'''"""FastAPI service that serves the persisted scikit-learn pipeline.

Run locally with:   uvicorn app:app --reload
"""
import os
import joblib                                    # to load the persisted pipeline
import numpy as np                               # to shape the incoming vector for sklearn
from fastapi import FastAPI, HTTPException       # web framework + typed error responses
from pydantic import BaseModel, Field            # request-body schema + validation
from typing import List

# ---- config baked in from the training notebook ----
N_FEATURES = {N_FEATURES}                          # the model expects EXACTLY this many features
MODEL_VERSION = "{MODEL_VERSION}"                  # bump this whenever you retrain / redeploy
MODEL_PATH = os.path.join(os.path.dirname(__file__), "model.joblib")

# ---- load the model ONCE at import/startup, not per request ----
# Loading is relatively expensive; doing it here means every /predict call is just a fast forward pass.
model = joblib.load(MODEL_PATH)

app = FastAPI(title="Breast Cancer Classifier", version=MODEL_VERSION)


class PredictRequest(BaseModel):
    """Request body schema. Pydantic validates this BEFORE our code runs."""
    # A flat list of floats == one sample's feature vector. The Field description shows up
    # in the auto-generated OpenAPI docs at /docs.
    features: List[float] = Field(..., description=f"Exactly {{N_FEATURES}} numeric features")


class PredictResponse(BaseModel):
    """Response schema -> gives clients a documented, typed contract."""
    prediction: int                              # predicted class label (0 or 1)
    probability: float                           # probability of the predicted class
    probabilities: List[float]                   # full per-class probability vector
    model_version: str                           # which model produced this answer


@app.get("/health")
def health():
    """Cheap liveness probe: confirms the process is up and the model is loaded."""
    return {{"status": "ok", "model_version": MODEL_VERSION, "n_features": N_FEATURES}}


@app.post("/predict", response_model=PredictResponse)
def predict(req: PredictRequest):
    """Score one feature vector and return the predicted class + probabilities."""
    # Length check: Pydantic guarantees floats, but not COUNT. Reject wrong-sized vectors with 422.
    if len(req.features) != N_FEATURES:
        raise HTTPException(
            status_code=422,
            detail=f"Expected {{N_FEATURES}} features, got {{len(req.features)}}",
        )
    # sklearn expects a 2D array of shape (n_samples, n_features); reshape our single row to (1, N).
    x = np.asarray(req.features, dtype=float).reshape(1, -1)
    proba = model.predict_proba(x)[0]            # per-class probabilities for this one sample
    pred = int(np.argmax(proba))                 # predicted class = the most probable one
    return PredictResponse(
        prediction=pred,
        probability=float(proba[pred]),          # confidence in the chosen class
        probabilities=[float(p) for p in proba], # full distribution, JSON-friendly floats
        model_version=MODEL_VERSION,
    )
'''

# Write the service to disk next to the model artifact.
APP_PATH = os.path.join(SERVICE_DIR, "app.py")
with open(APP_PATH, "w", encoding="utf-8") as f:
    f.write(app_source)
print("wrote", APP_PATH, f"({len(app_source)} chars)")

## 4. Exercise the API *in-process* with `TestClient`

This is the runnable proof. We import the `app` object from the `app.py` we just generated and wrap it in `TestClient`. `TestClient` speaks to the ASGI app directly in-memory — no socket, no port, no blocking `uvicorn.run`. So we get genuine HTTP semantics (status codes, JSON bodies, Pydantic validation) while the cell still returns instantly.

We hit `/health`, then `/predict` with a real row from the test set, assert `200`, and print the JSON.

In [ ]:
import sys, importlib
from fastapi.testclient import TestClient             # Starlette's TestClient, re-exported by FastAPI

# Make api_service importable, then import the module by name to get its `app` object.
if SERVICE_DIR not in sys.path:
    sys.path.insert(0, SERVICE_DIR)
import app as app_module                              # runs app.py top-to-bottom -> loads model.joblib
importlib.reload(app_module)                          # re-run cleanly if this cell is executed again

# TestClient drives app_module.app IN THIS PROCESS. `with` triggers FastAPI startup/shutdown events.
with TestClient(app_module.app) as client:
    # --- GET /health ---
    health = client.get("/health")
    assert health.status_code == 200, health.text     # must be alive
    print("GET /health ->", json.dumps(health.json()))

    # --- POST /predict with a REAL sample row from the held-out test set ---
    sample = X_test[0].tolist()                        # one 30-float feature vector
    resp = client.post("/predict", json={"features": sample})
    assert resp.status_code == 200, resp.text          # the load-bearing assertion
    result = resp.json()
    print("POST /predict ->", json.dumps(result, indent=2))

    # Cross-check: the API's answer matches calling the pipeline directly. Serving == training.
    print("direct pipeline predict:", int(pipeline.predict(X_test[:1])[0]),
          "| true label:", int(y_test[0]))

### Schema validation in action

Sending the wrong number of features should be *rejected*, not crash the server. Our endpoint returns `422 Unprocessable Entity` — the standard code FastAPI/Pydantic use for a request that is well-formed JSON but fails the schema/business rules.

In [ ]:
with TestClient(app_module.app) as client:
    # Deliberately send too few features -> our length guard raises HTTPException(422).
    bad = client.post("/predict", json={"features": [0.0, 1.0, 2.0]})
    print("bad request status:", bad.status_code)      # expect 422
    assert bad.status_code == 422, bad.text
    print("detail:", bad.json()["detail"])

    # Missing the 'features' key entirely -> Pydantic itself rejects it (also 422).
    missing = client.post("/predict", json={})
    print("missing-field status:", missing.status_code)
    assert missing.status_code == 422, missing.text
print("validation behaves as designed: malformed input -> 422, never a 500 crash")

## 5. Containerize: `requirements.txt` + `Dockerfile`

To run this service anywhere (a colleague's laptop, CI, the cloud) we package it in a Docker image. Two files do the job:

- **`requirements.txt`** pins the runtime dependencies so the image is reproducible.
- **`Dockerfile`** starts from a slim Python base, copies the service + model artifact in, installs the deps, and launches `uvicorn` as the container command.

We only *write and display* these files — building an image needs the Docker daemon and network, which we deliberately avoid so the notebook stays offline.

In [ ]:
# Pin the exact packages the container needs at runtime. joblib rides along with scikit-learn
# but we list it explicitly since app.py imports it directly.
requirements_txt = """fastapi
uvicorn[standard]
scikit-learn
numpy
joblib
pydantic
"""
REQ_PATH = os.path.join(SERVICE_DIR, "requirements.txt")
with open(REQ_PATH, "w", encoding="utf-8") as f:
    f.write(requirements_txt)
print("wrote", REQ_PATH)
print("-" * 50)
print(requirements_txt)

In [ ]:
# A minimal, layer-cache-friendly Dockerfile.
dockerfile = """# Slim base keeps the image small; 3.11 matches a modern sklearn/fastapi stack.
FROM python:3.11-slim

# All app files live under /app inside the container.
WORKDIR /app

# Copy requirements FIRST and install them. Docker caches this layer, so code-only
# changes don't force a full reinstall on every rebuild.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Now copy the service code and the persisted model artifact into the image.
COPY app.py .
COPY model.joblib .

# Document the port the app listens on.
EXPOSE 8000

# Launch the ASGI server. 'app:app' == the `app` object inside app.py.
# 0.0.0.0 makes it reachable from outside the container (not just localhost inside it).
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
DOCKER_PATH = os.path.join(SERVICE_DIR, "Dockerfile")
with open(DOCKER_PATH, "w", encoding="utf-8") as f:
    f.write(dockerfile)
print("wrote", DOCKER_PATH)
print("-" * 50)
print(dockerfile)

In [ ]:
# Final inventory: everything a deployment needs now lives in one folder.
print("contents of", SERVICE_DIR, ":")
for name in sorted(os.listdir(SERVICE_DIR)):
    full = os.path.join(SERVICE_DIR, name)
    print(f"  {name:20s} {os.path.getsize(full):>8d} bytes")

## 6. How to run it for real

The notebook proved the app works in-process. To serve it as an actual HTTP endpoint, use the generated files.

**Run locally (from inside `api_service/`):**
```bash
uvicorn app:app --reload
```
`--reload` auto-restarts on code edits (dev only). Interactive docs appear at `http://127.0.0.1:8000/docs`.

**Sample request with `curl`:**
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871, 1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193, 25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]}'
```

**Health check:**
```bash
curl http://127.0.0.1:8000/health
```

**Build and run the Docker image (from inside `api_service/`):**
```bash
docker build -t breast-cancer-api .
docker run -p 8000:8000 breast-cancer-api
```
`-p 8000:8000` maps the container's port to your host so `curl http://127.0.0.1:8000/...` reaches it.

---

### Production notes

- **Schema validation.** Pydantic rejects malformed bodies with `422` *before* your code runs; our explicit length check guards the feature count. Never trust raw request input — validate it into a typed model first.
- **Model versioning.** Every response carries `model_version`. Version your artifacts (`model_v1.0.0.joblib`), log which version served each prediction, and you can roll back or A/B test safely. The container also pins the version via the image tag.
- **Why `TestClient` here, not a live server.** `uvicorn app:app` (or `uvicorn.run(...)`) blocks the process forever waiting for connections — perfect for a real deployment, fatal for a notebook cell (it would hang and never finish). `TestClient` drives the identical ASGI `app` object in-process, giving real status codes and JSON with no networking and no blocking, so this notebook runs cleanly top-to-bottom. Use `TestClient` for tests/CI; use `uvicorn`/Docker to actually serve.